# Evaluate held-out RNN classification accuracy

Loads the trained models from `outputs/training/`, collects held-out predictions, averages performance across whatever seeds exist on disk (3 in demo mode, 10 in full mode), and writes the figure-ready outputs to `outputs/evaluation/`:

- `rnn_classification_accuracy_subset.mat` — MATLAB-compatible summary (per-array × per-category accuracy mean and SEM, trial counts, per-condition accuracy, chance levels, seed-overall accuracy).
- `evaluation_results.pkl` — same content as a Python pickle for downstream analyses.
- `movement_set_accuracy_mean_subset.csv`, `condition_accuracy_mean.csv` — CSV exports for spreadsheet inspection.
- `confusion_matrix_<array>.svg/.png` — example per-array confusion matrix figure.

The heatmap at the bottom is the Figure 3a layout. Cells flagged with a red ‘X’ failed the per-cell binomial test against the category's chance level.

> **Demo-mode note.** With a single trained seed there is no seed averaging and the SEM columns will be zero. Per-cell accuracy is still meaningful.

In [ ]:
import re
from whole_body_pipeline import OUTPUT_DIR, N_FOLDS
from neural_decoder_trainer import autoDetectDevice

DEVICE = autoDetectDevice()
formatted_data_dir = OUTPUT_DIR / 'formatted_data'
training_dir       = OUTPUT_DIR / 'training'
evaluation_dir     = OUTPUT_DIR / 'evaluation'

# Auto-detect the seeds present on disk. Take the intersection across all folds
# so we only evaluate seeds that completed for every fold.
def _seeds_in_fold(fold_idx):
    seed_dirs = (training_dir / f'fold_{fold_idx}_4sec').glob('seed=*')
    seeds = set()
    for d in seed_dirs:
        if (d / 'modelWeights').exists():
            m = re.search(r'seed=(\d+)', str(d))
            if m:
                seeds.add(int(m.group(1)))
    return seeds

per_fold_seeds = [_seeds_in_fold(f) for f in range(N_FOLDS)]
available_seeds = sorted(set.intersection(*per_fold_seeds)) if per_fold_seeds and all(per_fold_seeds) else []

print(f'Device:            {DEVICE}')
print(f'Formatted data:    {formatted_data_dir}')
print(f'Training output:   {training_dir}')
print(f'Evaluation output: {evaluation_dir}')
print(f'Seeds available across all folds: {available_seeds}')

if not available_seeds:
    raise RuntimeError(
        'No fully trained seeds found across all folds — run notebook 02 first.'
    )

In [ ]:
from whole_body_pipeline import evaluate_seeds, save_evaluation_outputs

results = evaluate_seeds(
    formatted_data_dir=formatted_data_dir,
    training_dir=training_dir,
    seeds=available_seeds,
    n_folds=N_FOLDS,
    batch_size=64,
    device=DEVICE,
    mode='subset',
)

save_evaluation_outputs(results, evaluation_dir)
print(f"Mean overall held-out accuracy across {len(available_seeds)} seeds: {results['seed_overall_accuracy'].mean():.3f}")
print(f'Saved evaluation outputs to {evaluation_dir}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from scipy.stats import binomtest

from whole_body_pipeline import RAW_DATA_DIR

ordered      = results['array_order_idx']
arrayOrder   = results['array_order']
cueSetNames  = results['movement_set_labels']
cueSets      = results['movement_sets']
dayAcc       = results['movement_accuracy_mean']
dayTrials    = results['movement_trial_counts']

movement_accuracy     = dayAcc[ordered, :]
movement_trial_counts = dayTrials[ordered, :]
chance_levels = np.array([1 / len(classes) for classes in cueSets])

# Per-cell significance: binomial test against chance
significant = np.zeros(movement_accuracy.shape, dtype=bool)
for row_idx in range(movement_accuracy.shape[0]):
    for col_idx in range(movement_accuracy.shape[1]):
        n_trials = int(movement_trial_counts[row_idx, col_idx])
        accuracy = movement_accuracy[row_idx, col_idx]
        if n_trials > 0 and np.isfinite(accuracy):
            n_correct = int(np.rint(n_trials * accuracy))
            significant[row_idx, col_idx] = (
                binomtest(n_correct, n_trials, chance_levels[col_idx], alternative='greater').pvalue < 0.05
            )

for label, chance in zip(cueSetNames, chance_levels):
    print(f'{label}: chance = {chance:.3f}')

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(movement_accuracy, vmin=0.2, vmax=1, cmap='viridis', aspect='auto')

# Cell-value annotations (white over viridis for readability)
for row_idx in range(movement_accuracy.shape[0]):
    for col_idx in range(movement_accuracy.shape[1]):
        val = movement_accuracy[row_idx, col_idx]
        if np.isfinite(val):
            ax.text(col_idx, row_idx, f'{val:.2f}',
                    ha='center', va='center', color='white', fontsize=8)

ax.set_yticks(np.arange(len(arrayOrder)))
ax.set_yticklabels(arrayOrder, fontsize=8)
ax.set_xticks(np.arange(len(cueSetNames)))
ax.set_xticklabels(cueSetNames, rotation=90, fontsize=8)
fig.colorbar(im, ax=ax, label='classification accuracy')

# Red X over non-significant cells
for row_idx in range(significant.shape[0]):
    for col_idx in range(significant.shape[1]):
        if movement_trial_counts[row_idx, col_idx] > 0 and not significant[row_idx, col_idx]:
            marker = ax.text(col_idx + 0.25, row_idx, 'X',
                             ha='right', va='center',
                             color='red', fontsize=13, fontweight='bold')
            marker.set_path_effects([path_effects.withStroke(linewidth=1.5, foreground='black')])

plt.tight_layout()

# Save into the pcg-mosaic Data/MainFigs/Fig3 folder, matching mainFigs.m output structure
fig3_dir = RAW_DATA_DIR / 'MainFigs' / 'Fig3'
fig3_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig3_dir / 'Fig3a_RNN_AccuracyHeatmap.png', dpi=200)
fig.savefig(fig3_dir / 'Fig3a_RNN_AccuracyHeatmap.svg')
print(f'Wrote {fig3_dir / "Fig3a_RNN_AccuracyHeatmap.png"} and .svg')

plt.show()

In [ ]:
for array_name, row in zip(results['array_order'], movement_accuracy):
    summary = ', '.join(
        f'{label}: {acc:.3f}'
        for label, acc in zip(results['movement_set_labels'], row)
    )
    print(f'{array_name}: {summary}')

## Example confusion matrix

Reproduces the per-array confusion matrix in Figure 3b for a single array (default `T12-v1`). Change `example_array_name` below to inspect a different array.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from whole_body_pipeline import (
    MOVEMENT_SETS,
    RAW_DATA_DIR,
    collect_seed_predictions,
    load_participant,
    matlab_string,
)

example_array_name = 'T12-v1'
fold_dataset_paths = [
    formatted_data_dir / f'all_fold_{fold_idx}_4sec.pkl'
    for fold_idx in range(N_FOLDS)
]

seed_scores = []
labels = None
arrays = None
for seed in available_seeds:
    scores, seed_labels, seed_arrays = collect_seed_predictions(
        fold_dataset_paths, training_dir, seed, batch_size=64, device=DEVICE
    )
    seed_scores.append(scores)
    if labels is None:
        labels = seed_labels
        arrays = seed_arrays
    else:
        assert np.array_equal(labels, seed_labels)
        assert np.array_equal(arrays, seed_arrays)

mean_scores      = np.mean(np.stack(seed_scores, axis=0), axis=0)
predicted_labels = np.argmax(mean_scores, axis=1)

example_array_idx = results['array_list'].index(example_array_name)

# Cross-validated average decoding accuracy for this array — out-of-fold
# predictions concatenated across folds and averaged across seeds.
mask = arrays == example_array_idx
n_correct = int((predicted_labels[mask] == labels[mask]).sum())
n_total   = int(mask.sum())
accuracy_pct = 100.0 * n_correct / n_total if n_total else float('nan')

confusion_matrix = np.zeros((mean_scores.shape[1], mean_scores.shape[1]))
for pred_label, true_label in zip(
    predicted_labels[mask],
    labels[mask],
):
    confusion_matrix[pred_label, true_label] += 1

col_sums = confusion_matrix.sum(axis=0, keepdims=True)
confusion_matrix = np.divide(
    confusion_matrix,
    col_sums,
    out=np.zeros_like(confusion_matrix),
    where=col_sums > 0,
)

cue_order = np.concatenate(MOVEMENT_SETS[1:])
participant = example_array_name.split('-')[0]
participant_data = load_participant(participant)
cue_labels = [
    matlab_string(participant_data['cueList'][cue_idx, 0]) for cue_idx in cue_order
]
example_matrix = confusion_matrix[np.ix_(cue_order, cue_order)]

SMALL_SIZE = 5
MEDIUM_SIZE = 6
BIGGER_SIZE = 7
plt.rc('font', size=SMALL_SIZE)
plt.rc('axes', titlesize=MEDIUM_SIZE)
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=SMALL_SIZE)
plt.rc('ytick', labelsize=SMALL_SIZE)
plt.rc('legend', fontsize=SMALL_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)
plt.rcParams['svg.fonttype'] = 'none'

plt.figure(figsize=(4.5, 4.5), dpi=300)
plt.imshow(example_matrix, vmin=0, vmax=1, cmap='viridis')
plt.gca().invert_yaxis()
plt.colorbar(fraction=0.046, pad=0.04)
plt.yticks(np.arange(len(cue_order)), labels=cue_labels)
plt.xticks(np.arange(len(cue_order)), labels=cue_labels, rotation=90)
plt.title(f'{example_array_name} — {accuracy_pct:.1f}% mean decoding accuracy')

# Save into the pcg-mosaic Data/MainFigs/Fig3 folder, matching mainFigs.m output structure
fig3_dir = RAW_DATA_DIR / 'MainFigs' / 'Fig3'
fig3_dir.mkdir(parents=True, exist_ok=True)
save_array_name = participant       # e.g. 'T12' from 'T12-v1' — matches mainFigs.m Fig3b naming
plt.savefig(fig3_dir / f'Fig3b_RNN_ConfusionMatrix_{save_array_name}.png', bbox_inches='tight', dpi=300)
plt.savefig(fig3_dir / f'Fig3b_RNN_ConfusionMatrix_{save_array_name}.svg', bbox_inches='tight')
print(f'Wrote {fig3_dir / f"Fig3b_RNN_ConfusionMatrix_{save_array_name}.png"} and .svg')

plt.show()